## Example 1: autocommand functionality test 1

In [ ]:
import autocommand

# Example 1: Basic function decoration and parser creation
print("Example 1: Basic function decoration and parser creation")

try:
    # Define a simple function to be decorated
    def greet(name='World', count: int = 1):
        """Greet someone a specified number of times"""
        results = []
        for i in range(count):
            results.append(f"Hello, {name}!")
        return results
    
    # Use autoparse to create an argument parser
    parser_func = autocommand.autoparse(greet)
    print(f"Created parser function: {type(parser_func).__name__}")
    print(f"Original function preserved: {greet.__name__}")
    print(f"Function docstring: {greet.__doc__}")
    
    # Test the original function still works
    result = greet("Alice", 2)
    print(f"Function result: {result}")
    assert result == ["Hello, Alice!", "Hello, Alice!"], result
    
    # The wrapper parses a command line into the function's arguments
    assert parser_func.func is greet
    assert parser_func(["--name", "Alice", "--count", "2"]) == result
    assert parser_func(["-n", "Bob"]) == ["Hello, Bob!"]  # short options come from the first letter
    assert parser_func([]) == ["Hello, World!"]  # defaults apply
    print("✓ Command lines parsed into greet() arguments")
    
    print("autocommand example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"autocommand example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: autocommand functionality test 2

In [ ]:
import autocommand

# Example 2: Function with type annotations and complex arguments
print("Example 2: Function with type annotations and complex arguments")

try:
    # Define a function with various argument types
    def process_data(filename: str, output_dir: str = '/tmp', 
                    verbose: bool = False, max_items: int = 100,
                    scale: float = 1.0):
        """Process data from a file with various options"""
        settings = {
            'filename': filename,
            'output_dir': output_dir,
            'verbose': verbose,
            'max_items': max_items,
            'scale': scale
        }
        
        if verbose:
            print("Processing with settings:", settings)
        
        return f"Processed {filename} -> {output_dir} (scale={scale}, max={max_items})"
    
    # Create parser and show it handles type annotations
    parser_func = autocommand.autoparse(process_data)
    print(f"Parser created for function with typed arguments")
    
    # Test the function directly
    result = process_data('test.txt', verbose=True, scale=2.5)
    print(f"Direct function call result: {result}")
    expected = "Processed test.txt -> /tmp (scale=2.5, max=100)"
    assert result == expected, result
    
    # Annotations become argparse types; a bool default of False becomes a flag
    assert parser_func(["test.txt", "--verbose", "--scale", "2.5"]) == expected
    assert parser_func(["data.csv", "-m", "5", "-o", "out"]) == "Processed data.csv -> out (scale=1.0, max=5)"
    try:
        parser_func(["test.txt", "--max_items", "many"])  # not an int
    except SystemExit as exit_error:
        assert exit_error.code == 2, exit_error.code  # argparse usage error
        print("✓ Invalid int argument rejected with exit code 2")
    else:
        raise AssertionError("--max_items many was accepted")
    
    print("autocommand example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"autocommand example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: autocommand functionality test 3

In [ ]:
import autocommand

# Example 3: smart_open utility for file handling
print("Example 3: smart_open utility for file handling")

try:
    import io
    import tempfile
    import os
    
    # Test smart_open functionality
    print("Testing smart_open utility...")
    
    # Create a temporary file for testing
    with tempfile.NamedTemporaryFile(mode='w', delete=False) as f:
        f.write("Hello from autocommand smart_open test!\nLine 2\nLine 3")
        temp_filename = f.name
    
    # Test reading with smart_open
    print("Reading file with smart_open:")
    with autocommand.smart_open(temp_filename, 'r') as f:
        content = f.read()
        print(f"Content read: {repr(content[:50])}...")
    assert content == "Hello from autocommand smart_open test!\nLine 2\nLine 3", repr(content)
    assert f.closed  # a file smart_open opened by name is closed afterwards
    
    # An already-open file object (such as sys.stdout) is passed through and left open
    stream = io.StringIO()
    with autocommand.smart_open(stream) as f:
        assert f is stream
        f.write("written through smart_open")
    assert not stream.closed and stream.getvalue() == "written through smart_open"
    print("smart_open passes open file objects such as sys.stdout through without closing them")
    
    # Clean up
    os.unlink(temp_filename)
    
    print("autocommand example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"autocommand example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: autocommand functionality test 4

In [ ]:
import autocommand
import sys

# Example 4: Demonstrating automain and module-level execution control
print("Example 4: Demonstrating automain and module-level execution control")

try:
    # Show how automain works for module execution
    def main_function(config_file: str = 'config.json', debug: bool = False):
        """Main entry point for a CLI application"""
        print(f"Running main function with config: {config_file}")
        if debug:
            print("Debug mode enabled")
        
        return {
            'config_loaded': config_file,
            'debug_mode': debug,
            'status': 'completed'
        }
    
    # Demonstrate how automain would work (without actually calling it)
    print("automain can be used to make functions runnable as CLI commands")
    print(f"Function signature: {main_function.__name__}{main_function.__annotations__}")
    print(f"Function defaults: config_file='{main_function.__defaults__[0] if main_function.__defaults__ else 'None'}'")
    
    # Test the function directly
    result = main_function('test.json', debug=True)
    print(f"Function result: {result}")
    assert result == {'config_loaded': 'test.json', 'debug_mode': True, 'status': 'completed'}, result
    
    # automain only runs the function when given "__main__"; for any other module name
    # it returns the function unchanged (here the module is not __main__)
    decorated = autocommand.automain("not_main")(main_function)
    assert decorated is main_function
    print("automain('not_main') returned the function unchanged")
    
    # automain needs the module name: using it as a bare decorator is an error
    from autocommand.automain import AutomainRequiresModuleError
    try:
        autocommand.automain(main_function)
    except AutomainRequiresModuleError:
        print("✓ Bare @automain without a module name rejected")
    else:
        raise AssertionError("automain accepted a function in place of a module name")
    
    print("autocommand example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"autocommand example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: autocommand functionality test 5

In [ ]:
import autocommand
import inspect

# Example 5: Advanced features and error handling
print("Example 5: Advanced features and error handling")

try:
    # Test autocommand with a function that has complex behaviors
    def advanced_cli(input_files: list, output_format: str = 'json',
                    recursive: bool = False, exclude_patterns: list = None):
        """Advanced CLI function with list arguments and complex defaults"""
        if exclude_patterns is None:
            exclude_patterns = ['*.tmp', '*.log']
        
        result = {
            'files_processed': len(input_files) if input_files else 0,
            'output_format': output_format,
            'recursive_mode': recursive,
            'excluded_patterns': exclude_patterns
        }
        
        print(f"Processing {result['files_processed']} files...")
        return result
    
    # Show that autoparse handles complex argument types
    print("Testing complex argument parsing...")
    parsed_func = autocommand.autoparse(advanced_cli)
    print(f"Function successfully parsed: {callable(parsed_func)}")
    assert parsed_func.func is advanced_cli
    
    # The parser exposes every parameter: positional without a default, options with one
    namespace = parsed_func.parser.parse_args(["input.txt", "-o", "xml", "-r"])
    assert namespace.output_format == "xml" and namespace.recursive is True, namespace
    assert namespace.exclude_patterns is None, namespace
    print(f"Parsed options: output_format={namespace.output_format}, recursive={namespace.recursive}")
    
    # Test with different argument combinations
    test_cases = [
        (['file1.txt', 'file2.txt'], 'xml', True, ['*.bak']),
        (['single.txt'], 'json', False, None)
    ]
    
    expected_results = [
        {'files_processed': 2, 'output_format': 'xml', 'recursive_mode': True, 'excluded_patterns': ['*.bak']},
        {'files_processed': 1, 'output_format': 'json', 'recursive_mode': False, 'excluded_patterns': ['*.tmp', '*.log']},
    ]
    for i, ((files, fmt, rec, exc), expected) in enumerate(zip(test_cases, expected_results), 1):
        result = advanced_cli(files, fmt, rec, exc)
        print(f"Test case {i}: {result}")
        assert result == expected, result
    
    # Show module-level information
    print(f"\nAutocommand module has {len(dir(autocommand))} total attributes")
    print(f"Main functions: {[name for name in dir(autocommand) if not name.startswith('_') and callable(getattr(autocommand, name))]}")
    
    print("autocommand example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"autocommand example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")